# Relatedness-regression estimators — 1kg_eur

Fits the estimator set in [`src/he_estimators.py`](../src/he_estimators.py) to
`08_accumulate.ipynb`'s merged cross-products. **This notebook no longer
implements the estimators itself.** It used to carry its own WLS and its own
five-estimator table with the old `h2_Ped_W25` / `h2_Ped_f` spellings; that
predates the module, which is now the reference implementation, is unit-tested
(`make test` in `src/`), and is what
`archive/analyses/model_ladder/ladder_cells.md` and the eur_D2 meta-analysis
were run with. Duplicating it here means two definitions of the same quantity
drifting apart, so the duplication is gone.

Three things change as a result, beyond the names.

**Standard errors are delete-block jackknife on the estimator**, not analytic
WLS SEs on bin means. Every estimator is refitted on each of the merge's
delete-block replicates (`.jk.tsv`), using the same blocks the accumulators
were built with. Differences like `diff_Rel-Unrel` are formed *inside* each
replicate, so their SEs carry the covariance of the two terms. With 50 blocks,
read `z` against t on 49 df.

**Fits are pair-weighted** (`w = sqrt(n)` on bin means), which is identical to
pair-level OLS with each pair's `a_ij` taken as its bin midpoint. The old code
weighted by `1/jk_se**2`, which is a different estimator — not a different API
for the same one.

**Class sets are the module's**, and they are not what a naive reading
suggests:

| label | classes summed | meaning |
|---|---|---|
| `noPO` | `other` + `FS` | **primary** — parent-offspring pairs removed |
| `pooled` | `other` + `FS` + `PO` | what an unclassified run would give |
| `PO` | `PO` | contrast, for the FS-range estimators |

Note `noPO` is *not* the `FS` class alone: it keeps every non-PO pair in the
range, including first-degree pairs KING did not label. And `pooled` here is
summed from the three real classes rather than read from the merged file's own
`pooled` row — the two agree unless some other class exists (see the class
check below).

Estimator names, ranges and models are documented in the module docstring and in
`archive/analyses/model_ladder/ladder_cells.md`. U = `a < 0.02`,
T = 0.02–0.05, R = 0.05–0.7. The offset models use degree bands starting at
deg3: deg3 0.09–0.18, deg2 0.18–0.36, deg1 0.36–0.7 (KING cutoffs doubled and
rounded to bin edges). The 4th-degree band 0.05–0.09 carries no offset and is
excluded from those fits — barely-related pairs there make the offset noise.

**Prerequisites:** `08_accumulate.ipynb` — `*_merged.full.tsv` **and**
`*_merged.jk.tsv` per combo, plus the bins file they were merged with.

## Setup

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"

REPO_DIR = os.environ.get("AOUCOV_REPO") or next(
    (p for p in (os.path.expanduser("~/repos/aou-covariance"),
                 os.path.expanduser("~/repos/AOU-covariance"),
                 os.path.expanduser("~/aou-covariance"))
     if os.path.isdir(os.path.join(p, "src"))),
    None,
)
assert REPO_DIR and os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"cannot find the repo checkout (REPO_DIR={REPO_DIR!r}) — set AOUCOV_REPO"
)

WORK_DIR = os.path.expanduser(f"~/grm_pheno_cov_{RUN_DIR}")

# Must be the bins file and block count 08_accumulate.ipynb merged with --
# load_grid turns the bin edges into the midpoints the fits regress on, and the
# jackknife replicates are indexed by block.
BINS_FILE = f"{WORK_DIR}/bins_wide.txt"
NBLOCKS   = 50

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}/07_estimators/he_regression"
PLOTS_DIR  = f"{BUCKET_DIR}/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)   # survives VM deletion, unlike WORK_DIR

sys.path.insert(0, f"{REPO_DIR}/src")
import importlib
import he_estimators as HE
importlib.reload(HE)
assert hasattr(HE, "DEG_BANDS"), "he_estimators.py predates the ladder -- git pull"

assert os.path.isfile(BINS_FILE), (
    f"missing {BINS_FILE} — 08_accumulate.ipynb writes it with "
    f"src/make_bins.py --wide; it must be the same file the merge used"
)
MID = HE.load_grid(BINS_FILE)

print(f"repo      : {REPO_DIR}")
print(f"work      : {WORK_DIR}")
print(f"bins      : {BINS_FILE}  ({len(MID)} bins)")
print(f"nblocks   : {NBLOCKS}")
print(f"classes   : {HE.CLS}   (noPO = {HE.NOPO})")
print(f"bucket    : {BUCKET_DIR}")

# ── shared style (matches 11_crossproduct_curves.ipynb) ───────────────────────
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#fcfcfb", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")
# validated categorical hues (all-pairs colourblind floors: CVD dE 9.2)
CLASS_COL = {"other": "#2a78d6", "FS": "#eb6834", "PO": "#1baf7a",
             "noPO": "#eb6834", "pooled": "#2a78d6"}
# covariate sets are ORDERED, so they get a sequential ramp, not a categorical
# palette -- this replaces viridis, which implied a magnitude that isn't there
RAMP = ["#86b6ef", "#3987e5", "#256abf", "#104281"]
FONT_STACK = ["Helvetica Neue", "Helvetica", "Liberation Sans", "Nimbus Sans",
              "Arial", "DejaVu Sans"]
from matplotlib import font_manager as _fm
_have = {x.name for x in _fm.fontManager.ttflist}
FONT = next((f for f in FONT_STACK if f in _have), "DejaVu Sans")
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "figure.dpi": 120,
    "font.family": FONT, "font.size": 9,
    "mathtext.fontset": "custom", "mathtext.rm": FONT,
    "mathtext.it": f"{FONT}:italic", "mathtext.bf": f"{FONT}:bold",
    # theme_bw / theme_linedraw style: a full panel border, and a standard grid
    # on both axes (major plus a lighter minor) instead of ad-hoc rules
    "axes.edgecolor": INK2, "axes.linewidth": 0.7, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.spines.top": True, "axes.spines.right": True,
    # sparingly: major gridlines only, light, with few of them. A minor grid
    # on both axes turned the panel into graph paper and competed with the data.
    "axes.grid": True, "axes.grid.which": "major", "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.5, "grid.alpha": 0.7,
    "xtick.minor.visible": False, "ytick.minor.visible": False,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "legend.frameon": False,
})
YLABEL = r"$\mathrm{E}[\tilde{y}_i\,\tilde{y}_j]$"   # y~ = standardised residual
print(f"font in use: {FONT}"
      f"   (of {FONT_STACK}, present: {[f for f in FONT_STACK if f in _have] or None})")


## Discover merged combos

Needs both halves of each merge: `.full.tsv` for the point estimates and
`.jk.tsv` for the jackknife replicates. A combo missing its `.jk.tsv` is
skipped rather than silently fitted without SEs.

The tag is parsed as the first three `__`-separated fields, so filenames
carrying extra provenance tags (the `__<class_tag>__<bin_tag>` convention used
in `archive/analyses/model_ladder/`) parse the same way as the plain
`{phenotype}__{transform}__{covariate_set}` that `08_accumulate.ipynb` writes.

In [ ]:
tags, skipped = [], []
for full in sorted(glob.glob(f"{WORK_DIR}/*_merged.full.tsv")):
    tag = os.path.basename(full)[: -len("_merged.full.tsv")]
    parts = tag.split("__")
    if len(parts) < 3:
        skipped.append((tag, "unparseable name"))
        continue
    if not os.path.isfile(f"{WORK_DIR}/{tag}_merged.jk.tsv"):
        skipped.append((tag, "no .jk.tsv"))
        continue
    tags.append(tag)

assert tags, (
    f"no merged combo with both .full.tsv and .jk.tsv under {WORK_DIR} — "
    f"run 08_accumulate.ipynb first (and check its RUN_DIR matches)"
)
print(f"{len(tags)} merged combos")
for t, why in skipped:
    print(f"  skipped {t}: {why}")

# Which classes the accumulate run produced. he_estimators only knows
# other/FS/PO -- anything else is silently dropped from every fit, so say so.
_probe = pd.read_csv(f"{WORK_DIR}/{tags[0]}_merged.full.tsv", sep="\t", usecols=["class"])
present = sorted(_probe["class"].unique())
print(f"\nclasses in the merged output: {present}")
missing = [c for c in HE.CLS if c not in present]
extra = [c for c in present if c not in HE.CLS + ["pooled"]]
if missing:
    print(f"  NOTE: {missing} absent — fits summing them fall back to what is there,"
          f"\n        and PO-contrast estimators will be NaN if PO is missing.")
if extra:
    print(f"  WARNING: {extra} not in he_estimators.CLS, so these pairs enter NO fit."
          f"\n           Dup/MZ pairs sit at a_ij ~ 1, outside every fit range, so this"
          f"\n           is harmless for the estimators -- but it does mean the module's"
          f"\n           `pooled` is not the merged file's `pooled` row.")

## Fit every estimator for every combo

`HE.jackknife` returns one row per `(estimator, classes)` with the full-data
point estimate and its delete-block SE. `est/se` is the `z` the ladder is read
with.

In [ ]:
parts = []
for i, tag in enumerate(tags, 1):
    S, N = HE.load_arrays(f"{WORK_DIR}/{tag}_merged.full.tsv",
                          f"{WORK_DIR}/{tag}_merged.jk.tsv", len(MID), NBLOCKS)
    r = HE.jackknife(S, N, MID, NBLOCKS)
    r["phenotype"], r["transform"], r["covset"] = tag.split("__")[:3]
    parts.append(r)
    if i % 25 == 0 or i == len(tags):
        print(f"  [{i}/{len(tags)}]")

E = pd.concat(parts, ignore_index=True)
E["z"] = E["est"] / E["se"]

out_tsv = f"{BUCKET_DIR}/estimators_{RUN_DIR}.tsv"
E.to_csv(out_tsv, sep="\t", index=False)
print(f"\n{len(E):,} rows ({E['estimator'].nunique()} estimators x "
      f"{E['classes'].nunique()} class sets x {len(tags)} combos)")
print(f"-> {out_tsv}")

TRANSFORMS = sorted(E["transform"].unique())
COVSETS = sorted(E["covset"].unique())
REF_TF = "invnorm" if "invnorm" in TRANSFORMS else TRANSFORMS[0]
REF_CS = "base_pcs" if "base_pcs" in COVSETS else COVSETS[0]
print(f"\ntransforms: {TRANSFORMS}\ncovsets:    {COVSETS}")
print(f"reference for the summary tables: {REF_TF} / {REF_CS}")
E.head(15)

## The slope ladder

All slopes pass through the origin. Read left to right: a single slope over
everything, then a separate slope for related pairs, then the same two with an
intercept offset allowed per degree band. `diff_Rel-RelOffsets` is how much of
the related slope the offsets absorb — a large value means the apparent extra
slope was really a level shift at each degree, which is what shared environment
looks like.

In [ ]:
LADDER_COLS = ["h2_Unrel", "h2_OneSlope", "h2_Rel", "h2_OneSlopeOffsets",
               "h2_RelOffsets", "diff_Rel-Unrel", "diff_RelOffsets-Unrel",
               "diff_Rel-RelOffsets"]

ladder = (E[(E["transform"] == REF_TF) & (E["covset"] == REF_CS)
            & (E["classes"] == "noPO")]
          .pivot_table(index="phenotype", columns="estimator", values="est"))
ladder = ladder[[c for c in LADDER_COLS if c in ladder.columns]]
print(f"{REF_TF} / {REF_CS}, classes = noPO\n")
ladder.round(3)

## PO vs FS — what the classification bought

`h2_FS` and `b2_FS` are fitted over 0.4–0.6 on each class set. `noPO` is the
answer with parent-offspring pairs removed; `pooled` is the pre-classification
number; `PO` is the contrast.

PO and FS pairs sit at the same `a_ij` but differ in what they share: FS pairs
share dominance (`d_ij = 0.25`) and usually a rearing environment, PO pairs
share neither. So `b2_FS` on `noPO` exceeding `b2_FS` on `PO` is the expected
direction, and the gap between `noPO` and `pooled` is how much the PO pairs
were diluting it. `excess` is the pair-weighted mean cross-product above the
additive prediction `h2_Unrel * a_ij`, reported per class by the module.

In [ ]:
ref = E[(E["transform"] == REF_TF) & (E["covset"] == REF_CS)]

fs_po = (ref[ref["estimator"].isin(["h2_FS", "b2_FS", "excess", "n_pairs"])]
         .pivot_table(index="phenotype", columns=["estimator", "classes"],
                      values="est"))
want = [(e, c) for e in ("n_pairs", "h2_FS", "b2_FS", "excess")
        for c in ("noPO", "pooled", "PO", "FS")]
fs_po = fs_po[[c for c in want if c in fs_po.columns]]

shift = None
if ("b2_FS", "noPO") in fs_po.columns and ("b2_FS", "pooled") in fs_po.columns:
    shift = (fs_po[("b2_FS", "noPO")] - fs_po[("b2_FS", "pooled")]).abs()
    print(f"|b2_FS(noPO) - b2_FS(pooled)|: median {shift.median():.4f}, "
          f"max {shift.max():.4f}")
    print("  (how far removing PO pairs moved the shared-environment estimate)\n")
fs_po.round(4)

## Model comparison

`HE.compare_models` scores the ladder rungs two ways for one combo:
`chi2_per_bin` is the full-data fit against each bin's own jackknife SE, and
`cv_error` fits each replicate on its own data and scores it on the pairs that
replicate left out — disjoint from what it was fitted on, so a richer model
cannot win by construction. `contrasts` gives the change in chi-square per bin
for each nested pair, with a jackknife SE, so `z` says whether the added term
earns its place.

In [ ]:
REF_TAG = next((t for t in tags
                if t.split("__")[1] == REF_TF and t.split("__")[2] == REF_CS), tags[0])
print(f"comparing models on: {REF_TAG}\n")

S, N = HE.load_arrays(f"{WORK_DIR}/{REF_TAG}_merged.full.tsv",
                      f"{WORK_DIR}/{REF_TAG}_merged.jk.tsv", len(MID), NBLOCKS)
fit_tbl, contrasts = HE.compare_models(S, N, MID, NBLOCKS)
print(fit_tbl.round(4).to_string(index=False))
print()
print(contrasts.round(4).to_string(index=False))

## Forest plots

One figure per `(phenotype, transform)`: estimate ±1 SE for each estimator,
coloured by covariate set, with the `noPO` and `pooled` class sets shown on
separate rows so the effect of removing PO pairs is visible on the same axis.
Estimators are ordered as in the ladder, with the FS-range ones first.

In [ ]:
PLOT_ESTIMATORS = ["h2_Unrel", "h2_FS", "b2_FS", "b2_step", "h2_PedW25",
                   "h2_Pedf", "h2_OneSlope", "h2_Rel", "h2_OneSlopeOffsets",
                   "h2_RelOffsets"]
PLOT_CLASSES = [c for c in ("noPO", "pooled", "PO") if c in set(E["classes"])]
CLASS_MARK = {"noPO": "o", "pooled": "s", "PO": "^"}

rows = [(e, c) for e in PLOT_ESTIMATORS for c in PLOT_CLASSES
        if not E[(E["estimator"] == e) & (E["classes"] == c)]["est"].isna().all()]
labels = [f"{e}  [{c}]" for e, c in rows]

saved = []
for (phe, tf), g in E.groupby(["phenotype", "transform"]):
    covsets = sorted(g["covset"].unique())
    colour = {cs: RAMP[i % len(RAMP)] for i, cs in enumerate(covsets)}

    fig, ax = plt.subplots(figsize=(7.5, 0.34 * len(rows) + 1.8))
    for j, cs in enumerate(covsets):
        d = (g[g["covset"] == cs].set_index(["estimator", "classes"])
             .reindex(rows))
        y = np.arange(len(rows)) + (j - (len(covsets) - 1) / 2) * 0.16
        ax.errorbar(d["est"], y, xerr=d["se"], fmt="o", ms=3.5, lw=1,
                    color=colour[cs], label=cs)

    ax.axvline(0, color="grey", lw=0.5)
    for i in range(1, len(rows)):
        if rows[i][0] != rows[i - 1][0]:
            ax.axhline(i - 0.5, color="0.9", lw=0.8)
    ax.set_yticks(np.arange(len(rows)))
    ax.set_yticklabels(labels, fontsize=7.5)
    ax.invert_yaxis()
    ax.grid(axis="x", lw=0.6, color=GRID); ax.set_axisbelow(True)
    ax.set_xlabel("estimate ±1 jackknife SE")
    ax.set_title(f"{phe} ({tf})")
    ax.legend(title="covariate set", fontsize=7.5, title_fontsize=8)
    plt.tight_layout()

    png = f"{PLOTS_DIR}/{phe}__{tf}__estimators.png"
    fig.savefig(png, dpi=150)
    plt.show()
    plt.close(fig)
    saved.append(png)

print(f"{len(saved)} plots saved to {PLOTS_DIR}")

## Export for the plotting notebooks

Two tables, written here rather than in a plotting notebook so that anything
which only draws — `11r_crossproduct_curves.ipynb` in particular — depends on
this notebook alone:

- **`curves_<run>.tsv`** — per `(combo, class set, bin)`: the pair-weighted mean
  cross-product, the pair count, and the per-bin delete-block jackknife SE.
  Computed by `he_estimators.bin_curve`, the same function
  `11_crossproduct_curves.ipynb` plots through, so no two figures can disagree
  about a bin mean or an SE. **Unfiltered**: every bin any class populates is
  here, including one-pair bins. Dropping sparse bins is a display choice and
  belongs to whoever draws.
- **`curve_params_<run>.tsv`** — the structural constants a renderer needs to
  place the ranges and reconstruct the fitted lines: `U`/`T`/`R` boundaries, the
  degree bands, the `h2_Pedf` breaks, the FS range, and `nblocks`. Not display
  settings — those live in the notebook doing the drawing.

In [ ]:
CURVE_OUT = (f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}"
             f"/07_estimators/crossproduct_curves")
os.makedirs(CURVE_OUT, exist_ok=True)

CLASS_SETS = {"noPO": HE.NOPO, "pooled": HE.POOLED, "PO": ["PO"]}

rows = []
for tag in tags:
    phe, tf, cs = tag.split("__")[:3]
    S, N = HE.load_arrays(f"{WORK_DIR}/{tag}_merged.full.tsv",
                          f"{WORK_DIR}/{tag}_merged.jk.tsv", len(MID), NBLOCKS)
    # class sets, then the single classes the figures plot separately
    for label, classes in {**CLASS_SETS, **{c: [c] for c in HE.CLS}}.items():
        a, m, n, se = HE.bin_curve(S, N, classes, MID, NBLOCKS)
        if not len(a):
            continue
        rows.append(pd.DataFrame({"phenotype": phe, "transform": tf, "covset": cs,
                                  "class_set": label, "bin_mid": a, "mean": m,
                                  "n_pairs": n, "se": se}))

curves = pd.concat(rows, ignore_index=True)
curves_path = f"{CURVE_OUT}/curves_{RUN_DIR}.tsv"
curves.to_csv(curves_path, sep="\t", index=False)

params = pd.DataFrame([
    ("run", RUN_DIR), ("nblocks", NBLOCKS), ("n_bins", len(MID)),
    ("u_hi", HE.U_HI), ("t_hi", HE.T_HI), ("r_hi", HE.R_HI),
    ("deg_lo", HE.DEG_LO),
    ("deg_bands", ";".join(f"{k}:{lo},{hi}" for k, (lo, hi) in HE.DEG_BANDS.items())),
    ("pedf_breaks", "0.1,0.2;0.2,0.4;0.4,0.6"),
    ("fs_range", "0.4,0.6"),
    ("ref_transform", REF_TF), ("ref_covset", REF_CS),
], columns=["key", "value"])
params_path = f"{CURVE_OUT}/curve_params_{RUN_DIR}.tsv"
params.to_csv(params_path, sep="\t", index=False)

print(f"{len(curves):,} rows -> {curves_path}")
print(f"{len(params)} parameters -> {params_path}")
print(f"class sets: {sorted(curves['class_set'].unique())}")
curves.head()

## Copy notebook to bucket

In [ ]:
import subprocess

_nb = os.path.join(REPO_DIR, "notebooks/09_estimators.ipynb")
_gs = ("gs://cloned-shared-env-pilot-wb-swift-sprout-7231/"
       f"{PROJECT_DIR}/{RUN_DIR}/notebooks/09_estimators.ipynb")
subprocess.run(["gcloud", "storage", "cp", _nb, _gs], check=True)
print(f"notebook -> {_gs}")